# Analisis Exploratorio de Datos (EDA) - Bank Marketing (UCI)
## Proyecto Final de Ciencia de Datos
### Centro Universitario UAEM Texcoco

Objetivo: Comprender la estructura del dataset, detectar valores faltantes implicitos, analizar el desbalance de clases, identificar variables con fuga de datos y extraer patrones clave previos al modelado.

In [ ]:
# 1. Cargar librerias
import pandas as pd # carga de datos
import numpy as np # analisis de datos
import matplotlib.pyplot as plt # graficas
import seaborn as sns
from pathlib import Path

# Configuracion estetica
sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
print("Librerias cargadas correctamente")

In [ ]:
# 2. Carga e inspeccion inicial
ruta_datos = Path("..") / "data" / "raw" / "bank-full.csv"
df = pd.read_csv(ruta_datos, sep=";")
print("carga de datos exitosa")
print("Dimensiones del dataset:", df.shape)
df.head()

In [ ]:
# Tipos de datos y revision de nulos/duplicados
df.info()
print("\nValores nulos explicitos (NaN):", df.isnull().sum().sum())
print("Filas duplicadas:", df.duplicated().sum())

In [ ]:
# Estadisticas descriptivas de variables numericas y categoricas
df.describe(include="all")

## 3. Revision de valores faltantes implicitos ('unknown') y desbalance de clases

In [ ]:
# Conteo de 'unknown' en variables categoricas
cat_vars = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]

unknown_datos = []
for col in cat_vars:
    cantidad = (df[col] == "unknown").sum()
    if cantidad > 0:
        unknown_datos.append({
            "variable": col,
            "cantidad_unknown": cantidad,
            "porcentaje": round((cantidad / len(df)) * 100, 2)
        })

df_unknown = pd.DataFrame(unknown_datos).sort_values("porcentaje", ascending=False)
print(df_unknown)

# Variable objetivo binaria para analisis
df["y_bin"] = (df["y"] == "yes").astype(int)
print("\nDistribucion de la clase objetivo (y):")
print(df["y"].value_counts())
print((df["y"].value_counts(normalize=True) * 100).round(2))

## 4. Analisis Univariado (Graficas)

In [ ]:
# Grafica 1: Variable objetivo (Desbalance de clases)
ax = sns.countplot(x="y", data=df)
plt.title("Distribucion de la variable objetivo (Contratacion de deposito)")
plt.xlabel("Contrato deposito a plazo (y)")
plt.ylabel("Cantidad de clientes")

for p in ax.patches:
    pct = f"{100 * p.get_height() / len(df):.2f}%"
    ax.annotate(pct, (p.get_x() + p.get_width() / 2., p.get_height()), ha="center", va="bottom")

plt.show()

In [ ]:
# Grafica 2: Porcentaje de valores 'unknown'
ax = sns.barplot(x="variable", y="porcentaje", data=df_unknown)
plt.title("Porcentaje de valores 'unknown' por variable categorica")
plt.xlabel("Variable")
plt.ylabel("Porcentaje (%)")

for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()), ha="center", va="bottom")

plt.show()

In [ ]:
# Grafica 3: Distribuciones de variables numericas clave
num_vars = ["age", "balance", "duration", "campaign"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flatten(), num_vars):
    sns.histplot(df[col], kde=True, bins=30, ax=ax)
    ax.set_title(f"Distribucion de {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

## 5. Analisis Bivariado y Correlaciones

In [ ]:
# Grafica 4: Tasa de contratacion segun poutcome, housing y job
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(x="poutcome", y="y_bin", data=df, ax=axes[0], errorbar=None)
axes[0].set_title("Tasa de aceptacion por poutcome")
axes[0].set_ylabel("Proporcion de 'yes'")

sns.barplot(x="housing", y="y_bin", data=df, ax=axes[1], errorbar=None)
axes[1].set_title("Tasa de aceptacion por hipoteca")
axes[1].set_ylabel("")

orden_job = df.groupby("job")["y_bin"].mean().sort_values(ascending=False).index
sns.barplot(y="job", x="y_bin", data=df, order=orden_job, ax=axes[2], errorbar=None)
axes[2].set_title("Tasa de aceptacion por ocupacion")
axes[2].set_xlabel("Proporcion de 'yes'")
axes[2].set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
# Grafica 5: Mapa de correlaciones
num_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous", "y_bin"]
corr = df[num_cols].corr()

sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Mapa de correlaciones (Variables numericas y objetivo)")
plt.show()